### GOAL 
Notebook aims to disentangle HGSOC RNA data into 2+ data spaces (Cell Cycle Signal Space, Inteferon Signal Space, Cell Dissociation Signal Space, Cell Type Signal Space)

In [1]:
import sys
import os

# Get the directory containing the notebook
NOTEBOOK_DIR = os.getcwd()

# Compute project root (go up one level)
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))

# Add project root to path
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

In [3]:
from src.data.umi_data import UMIVaeDataset
from src.celluntangler import utils
from src.celluntangler.models import Trainer
from src.celluntangler.models.nb_vae import NBVAE
from src.visualization.helpers import split_embeddings
from src.visualization.visualization_functions import visualize_poincare_from_lorentz, compute_umap

In [6]:
adata = sc.read_h5ad("../../../data/HGSOC/MALIGNANT_NONMALIGNANT_EPITHELIAL/epithelial.h5ad")

In [21]:
adata.obs['Motility.and.migration.module']

SPECTRUM-OV-107_S1_CD45N_RIGHT_ADNEXA_AAACCCACACTGGAAG        -0.107080
SPECTRUM-OV-107_S1_CD45N_RIGHT_ADNEXA_AAACGAAGTTCGAGCC        -0.147109
SPECTRUM-OV-107_S1_CD45N_RIGHT_ADNEXA_AAAGGATAGAGTACCG        -0.188379
SPECTRUM-OV-107_S1_CD45N_RIGHT_ADNEXA_AAAGGGCTCGTCCTTG        -0.267798
SPECTRUM-OV-107_S1_CD45N_RIGHT_ADNEXA_AAAGGTAAGGGCAGTT        -0.084868
                                                                 ...   
SPECTRUM-OV-118_S1_CD45N_PELVIC_PERITONEUM_TTTGGTTTCTGGGCCA    0.020611
SPECTRUM-OV-118_S1_CD45N_PELVIC_PERITONEUM_TTTGTTGAGGTCTGGA   -0.051175
SPECTRUM-OV-118_S1_CD45N_PELVIC_PERITONEUM_TTTGTTGCACGTCTCT   -0.162688
SPECTRUM-OV-118_S1_CD45N_PELVIC_PERITONEUM_TTTGTTGCACTTGAAC   -0.200266
SPECTRUM-OV-118_S1_CD45N_PELVIC_PERITONEUM_TTTGTTGCAGAAGCGT    0.158831
Name: Motility.and.migration.module, Length: 210107, dtype: float32

In [19]:
adata.obs.columns.to_numpy()

array(['percent.mt', 'percent.rb', 'doublet', 'sample', 'batch',
       'patient_id', 'S.Score', 'G2M.Score', 'Phase', 'CC.Diff',
       'RNA_snn_res.0.3', 'seurat_clusters', 'nCount_RNA', 'nFeature_RNA',
       'doublet_score', 'cell_id', 'cell_type_super', 'RNA_snn_res.0.1',
       'RNA_snn_res.0.2', 'cluster_label', 'CD8.Cytotoxic.module',
       'CD8.Dysfunctional.module', 'CD8.Naive.module',
       'CD8.Predysfunctional.module', 'CIN.module',
       'CIN.responsive.Noncanonical.NFKB.targets.up.module',
       'CIN.responsive.Noncanonical.NFKB.targets.down.module',
       'Noncanonical.NFKB.regulators.up.module',
       'Noncanonical.NFKB.regulators.down.module',
       'Canonical.NFKB.regulators.module', 'Interferon.regulators.module',
       'EMT.module', 'Inflammation.module',
       'Motility.and.migration.module', 'ISG.module', 'SASP.module',
       'Myofibroblast.module', 'Fibroblast.module',
       'Mesenchymal.stem.cell.module', 'Pericytes.module',
       'Smooth.muscle.cel

In [16]:
np.unique(adata.obs.tissue.to_numpy())

array(['abdomen', 'abdominal wall', 'adnexa of uterus', 'ascitic fluid',
       'caecum', 'diaphragm', 'fallopian tube', 'intestine',
       'large intestine', 'left ovary', 'liver', 'lymph node', 'omentum',
       'paracolic gutter', 'parietal peritoneum', 'peritoneum',
       'right ovary', 'transverse colon', 'urinary bladder'], dtype=object)

### Pre-processing

In [5]:
adata.var["gene_symbols"] = adata.var["feature_name"]

In [6]:
cell_cycle_genes_path = "../../../genes/celluntangler_human_cell_cycle_genes.tsv"
interferon_genes_path = "../../../genes/HGSOC/human_interferon_genes.tsv" 
dissociation_genes_path = "../../../genes/HGSOC/human_cell_dissociation_genes.tsv" 

cell_cycle_genes = pd.read_csv(cell_cycle_genes_path, header = None, sep="\t")
interferon_genes = pd.read_csv(interferon_genes_path, header = None, sep="\t")
dissociation_genes = pd.read_csv(dissociation_genes_path, header = None, sep = "\t")

In [7]:
## Brief Analysis of Number of Relevant Genes Found 

In [7]:
cell_cycle_genes_set = set(cell_cycle_genes.iloc[:,0])
interferon_genes_set = set(interferon_genes.iloc[:,0]) 
dissociation_genes_set = set(dissociation_genes.iloc[:,0])

In [8]:
gene_sets = { "cell cycle genes": cell_cycle_genes_set, 
              "interferon genes": interferon_genes_set,
              "dissociation_genes": dissociation_genes_set }
for name0, genes0 in gene_sets.items():  
    for name1, genes1 in gene_sets.items() : 
        if (name0 != name1): 
            print(f"Intersection between: {name0}, {name1}") 
            print(genes0.intersection(genes1))

Intersection between: cell cycle genes, interferon genes
set()
Intersection between: cell cycle genes, dissociation_genes
set()
Intersection between: interferon genes, cell cycle genes
set()
Intersection between: interferon genes, dissociation_genes
set()
Intersection between: dissociation_genes, cell cycle genes
set()
Intersection between: dissociation_genes, interferon genes
set()


In [9]:
contained_genes_cc = adata.var["gene_symbols"].isin(cell_cycle_genes[0])
print(f"Number of cell cycle genes present in adata: {np.sum(contained_genes_cc)}")

contained_genes_interferon = adata.var["gene_symbols"].isin(interferon_genes[0])
print(f"Number of interferon genes present in adata: {np.sum(contained_genes_interferon)}")

contained_genes_dissociation = adata.var["gene_symbols"].isin(dissociation_genes[0])
print(f"Number of cell dissociation genes present in adata: {np.sum(contained_genes_dissociation)}")

Number of cell cycle genes present in adata: 214
Number of interferon genes present in adata: 12
Number of cell dissociation genes present in adata: 17


In [10]:
# Order Genes in adata such that genes appear in following order: cell cycle genes, interferon genes, dissociation genes

cc_indices = np.where(contained_genes_cc)[0]
interferon_indices = np.where(contained_genes_interferon)[0]
dissociation_indices = np.where(contained_genes_dissociation)[0]

all_known_indices = np.hstack((cc_indices, interferon_indices, dissociation_indices))
remaining_indices = np.setdiff1d(np.arange(adata.shape[1]), all_known_indices)

rearranged_indices = np.hstack((cc_indices, interferon_indices, dissociation_indices, remaining_indices))
rearranged_indices = rearranged_indices.astype(int)

In [11]:
rearranged_indices

array([  146,   241,   501, ..., 31090, 31091, 31092], shape=(31093,))

In [12]:
print(cc_indices.shape, interferon_indices.shape, dissociation_indices.shape, remaining_indices.shape)

(214,) (12,) (17,) (30850,)


In [13]:
adata = adata[:, rearranged_indices] # .copy() is safer in AnnData
adata.uns["new_gene_ordering"] = rearranged_indices # NB: this takes a while to run!

/var/folders/2z/f9h5sjqj6m9dz0ljc7yy_plw0000gn/T/ipykernel_47914/1082852305.py:2: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["new_gene_ordering"] = rearranged_indices # NB: this takes a while to run!
